# General settings

In [1]:
import os

# You can only use one GPU at a time
os.environ["CUDA_VISIBLE_DEVICES"] = "7"
os.environ["OMP_NUM_THREADS"] = "32"
os.environ["MKL_NUM_THREADS"] = "32"

In [2]:
from vv.metrics.cmmd import CMMDMetric
import torch
from torch.utils.data import DataLoader, TensorDataset
import lightning as L
from lightning import Callback
import torch.nn as nn

./venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Data

In [3]:
# Define the shape of the images (e.g., 3 channels, 224x224 resolution)
image_shape = (3, 256, 256)
batch_size = 15
num_batches = 1000

# Generate random image-shaped tensors
dataset_1 = torch.rand(batch_size * num_batches, *image_shape)
tensor_dataset_1 = TensorDataset(dataset_1)

class ImageDataModule(L.LightningDataModule):
    def __init__(self, dataset, batch_size):
        super().__init__()
        self.dataset = dataset
        self.batch_size = batch_size

    def setup(self, stage=None):
        # Split the dataset into train, val, and test (using the same data for simplicity)
        self.train_dataset = self.dataset
        self.val_dataset = self.dataset
        self.test_dataset = self.dataset

    def train_dataloader(self):
        return DataLoader(self.train_dataset, batch_size=self.batch_size, shuffle=True)

    def val_dataloader(self):
        return DataLoader(self.val_dataset, batch_size=self.batch_size, shuffle=False)

    def test_dataloader(self):
        return DataLoader(self.test_dataset, batch_size=self.batch_size, shuffle=False)

# Create the data module
data_module = ImageDataModule(tensor_dataset_1, batch_size=batch_size)
data_module.setup()

# Model

In [4]:
# Define a dummy model
class DummyModel(L.LightningModule):
    def __init__(self):
        super(DummyModel, self).__init__()
        self.fc = nn.Conv2d(image_shape[0], image_shape[0], kernel_size=1)  # 1x1 convolution

    def forward(self, x):
        return self.fc(x)

    def training_step(self, batch, batch_idx):
        y = self(batch[0])
        loss = nn.functional.mse_loss(y, batch[0])  # Example: reconstruction loss
        return {"loss": loss, "rec": y}

    def validation_step(self, batch, batch_idx):
        y = self(batch[0])
        loss = nn.functional.mse_loss(y, batch[0])  # Example: reconstruction loss
        return {"loss": loss, "rec": y}

    def test_step(self, batch, batch_idx):
        y = self(batch[0])
        loss = nn.functional.mse_loss(y, batch[0])  # Example: reconstruction loss
        return {"loss": loss, "rec": y}
    
    def predict_step(self, batch, batch_idx):
        y = self(batch[0])
        return {"rec": y}
    
    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=0.001)
        return optimizer

# Example usage
dummy_model = DummyModel()

# CMMD Callback

In [5]:
class CMMDCallback(Callback):
    def __init__(self):
        super().__init__()
        self.cmmd_metric = CMMDMetric()

    def on_train_batch_end(self, trainer, pl_module, outputs, batch, batch_idx, dataloader_idx):
        input = batch
        output = outputs["rec"]
        self.cmmd_metric.update(input, output)

    def on_validation_batch_end(self, trainer, pl_module, outputs, batch, batch_idx, dataloader_idx):
        input = batch
        output = outputs["rec"]
        self.cmmd_metric.update(input, output)

    def on_test_batch_end(self, trainer, pl_module, outputs, batch, batch_idx, dataloader_idx):
        input = batch
        output = outputs["rec"]
        self.cmmd_metric.update(input, output)

    def on_predict_batch_end(self, trainer, pl_module, outputs, batch, batch_idx, dataloader_idx):
        input = batch
        output = outputs["rec"]
        self.cmmd_metric.update(input, output)

    def on_train_epoch_end(self, trainer, pl_module):
        cmmd_value = self.cmmd_metric.compute()
        print(f"CMMD metric: {cmmd_value.item():.10f}")
        self.cmmd_metric.reset()

    def on_validation_epoch_end(self, trainer, pl_module):
        cmmd_value = self.cmmd_metric.compute()
        print(f"CMMD metric: {cmmd_value.item():.10f}")
        self.cmmd_metric.reset()

    def on_test_epoch_end(self, trainer, pl_module):
        cmmd_value = self.cmmd_metric.compute()
        print(f"CMMD metric: {cmmd_value.item():.10f}")
        self.cmmd_metric.reset()

    def on_predict_epoch_end(self, trainer, pl_module):
        cmmd_value = self.cmmd_metric.compute()
        print(f"CMMD metric: {cmmd_value.item():.10f}")
        self.cmmd_metric.reset()

# Prepare Training

In [6]:
# Create the CMMD callback
cmmd_callback = CMMDCallback()

# Create the PyTorch Lightning trainer
trainer = L.Trainer(
    accelerator="cuda",
    devices=1,
    max_epochs=5,
    callbacks=[cmmd_callback],
    profiler="simple",
)

./venv/lib/python3.11/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `Clip MMD` will save all extracted features in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


# Run Training

In [7]:
print(dummy_model)
print(data_module)

DummyModel(
  (fc): Conv2d(3, 3, kernel_size=(1, 1), stride=(1, 1))
)
{Train dataloader: size=15000}
{Validation dataloader: size=15000}
{Test dataloader: size=15000}
{Predict dataloader: None}


In [8]:
trainer.fit(dummy_model, data_module)

You are using a CUDA device ('NVIDIA A100-PCIE-40GB') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [7]

  | Name | Type   | Params | Mode 
----------------------------------------
0 | fc   | Conv2d | 12     | train
----------------------------------------
12        Trainable params
0         Non-trainable params
12        Total params
0.000     Total estimated model params size (MB)
1         Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

./venv/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=63` in the `DataLoader` to improve performance.


Sanity Checking DataLoader 0:   0%|          | 0/2 [00:00<?, ?it/s]

TypeError: CMMDCallback.on_validation_batch_end() missing 1 required positional argument: 'dataloader_idx'